# FloraScan - Retrain Model for Higher Accuracy
Target: 89-93% validation accuracy

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import os
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Configuration
DATA_DIR = r'd:\Florascann\dataset_limited'
MODEL_PATH = r'd:\Florascann\models\model_v4_final.h5'
SAVE_PATH = r'd:\Florascann\models\model_v5_retrained.h5'

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20  # More epochs with early stopping

In [ ]:
# Load existing model
print("Loading saved model...")
model = load_model(MODEL_PATH)
print("Model loaded!")
model.summary()

In [ ]:
# Data generators with strong augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Classes: {train_generator.num_classes}")

In [ ]:
# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model recompiled with learning rate: 1e-5")

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# Train!
print("="*50)
print("RETRAINING FOR HIGHER ACCURACY")
print("Target: 89-93%")
print("="*50)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Final evaluation
test_loss, test_acc = model.evaluate(test_generator, verbose=0)

print("\n" + "="*50)
print("RETRAINED MODEL RESULTS")
print("="*50)
print(f"Final Validation Accuracy: {test_acc*100:.2f}%")
print(f"Final Validation Loss: {test_loss:.4f}")
print("="*50)

if test_acc >= 0.89:
    print("\n TARGET ACHIEVED! 89%+ accuracy!")
else:
    print(f"\n Current: {test_acc*100:.1f}% - Run more epochs if needed")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(r'd:\Florascann\results\retrain_history.png', dpi=150)
plt.show()

In [ ]:
# Save final model
model.save(SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}")

## Continue Training (Run again if needed)
If accuracy is still below 89%, run the training cell again!

In [ ]:
# Run this to continue training for more epochs
extra_history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_acc = model.evaluate(test_generator, verbose=0)
print(f"\nNew Accuracy: {test_acc*100:.2f}%")